In [ ]:
import os
import email
from email import policy
from email.header import decode_header
import pandas as pd
from tqdm import tqdm
import boto3
from dotenv import load_dotenv
from utils import *
import missingno as msno

In [ ]:
def rebuild_master_metadata(eml_dir, attach_dir, csv_path="yahoo_folders.csv"):
  records = []
  
  # Load master folder mapping to accurately resolve names with underscores
  if os.path.exists(csv_path):
    df_folders = pd.read_csv(csv_path)
    valid_folders = df_folders["folder_name"].dropna().tolist()
  else:
    valid_folders = []

  eml_files = [f for f in os.listdir(eml_dir) if f.endswith(".eml")]

  for filename in tqdm(eml_files, desc="Parsing local .eml files", unit="file"):
    try:
      # Strip prefix and extension, leaving safe_folder_prefix + uid
      core_name = filename.replace("email_", "").replace(".eml", "")
      
      # Match against known folders from CSV to safely handle literal underscores
      matched_folder = "Unknown"
      uid = ""
      
      for f_name in valid_folders:
        safe_f_prefix = "".join(c if c.isalnum() else "_" for c in f_name)
        prefix_check = f"{safe_f_prefix}_"
        if core_name.startswith(prefix_check):
          matched_folder = f_name
          uid = core_name[len(prefix_check):]
          break
          
      # Fallback if CSV lookup fails
      if not uid:
        parts = core_name.split("_")
        uid = parts[-1]
        matched_folder = " ".join(parts[:-1])

      eml_path = os.path.join(eml_dir, filename)
      with open(eml_path, "rb") as f:
        raw_email = f.read()

      msg = email.message_from_bytes(raw_email, policy=policy.default)

      subject = decode_str(msg.get("Subject"))
      sender = decode_str(msg.get("From"))
      recipient = decode_str(msg.get("To"))
      cc = decode_str(msg.get("Cc"))
      bcc = decode_str(msg.get("Bcc"))
      date = decode_str(msg.get("Date"))

      body = extract_body(msg)
      snippet = clean_snippet(body, max_len=500)

      has_attachments = 0
      attachment_names_list = []
      attachment_extensions_set = set()
      email_attach_dir = os.path.join(attach_dir, filename.replace(".eml", ""))

      if os.path.exists(email_attach_dir):
        for attach_file in os.listdir(email_attach_dir):
          has_attachments = 1
          attachment_names_list.append(attach_file)
          _, ext = os.path.splitext(attach_file)
          if ext:
            attachment_extensions_set.add(ext.lower())

      extensions_str = (
          ", ".join(sorted(attachment_extensions_set))
          if attachment_extensions_set
          else None
      )

      records.append({
          "folder": matched_folder,
          "uid": uid,
          "sender": sender,
          "recipient": recipient,
          "cc": cc if cc else None,
          "bcc": bcc if bcc else None,
          "date": date,
          "subject": subject,
          "has_attachments": has_attachments,
          "attachment_names": str(attachment_names_list),
          "attachment_extensions": extensions_str,
          "eml_path": eml_path,
          "body_snippet": snippet,
      })
    except Exception as e:
      tqdm.write(f"Error parsing {filename}: {e}")

  return pd.DataFrame(records)

In [ ]:
EML_DIR = "yahoo_local_archive/emls"
ATTACH_DIR = "yahoo_local_archive/attachments"

In [ ]:
# Execute reconstruction
yahoo_metadata_all = rebuild_master_metadata(EML_DIR, ATTACH_DIR)
yahoo_metadata_all.to_csv("yahoo_metadata_all.csv", index=False, encoding="utf-8")
print(f"✅ Successfully rebuilt metadata for {len(yahoo_metadata_all)} emails!")

In [ ]:
# Remote storage
load_dotenv()
bucket_name = os.getenv('AWS_BUCKET_NAME')

if not bucket_name:
    raise ValueError("❌ Error: 'AWS_BUCKET_NAME' is missing from your environment variables or .env file.")

print(f"AWS Bucket configured: {bucket_name}")

# s3 client
s3_client = boto3.client("s3")

In [ ]:
s3_client.upload_file("yahoo_metadata_all.csv", bucket_name, "yahoo_metadata_all.csv")

In [ ]:
# Visual check for the absence of missing metadata
msno.matrix(yahoo_metadata_all)